# Quantificando a Herança de Design Compartilhada em uma Frota de Transformadores de Potência com o PROC INBREED

## Resumo Executivo

Os transformadores de subestação de uma operadora de rede são projetados em gerações sucessivas de design, cada novo modelo derivado de dois designs predecessores. Este notebook trata essa genealogia de engenharia como um pedigree e usa o **PROC INBREED** para calcular coeficientes de endogamia e de parentesco aditivo (covariância), quantificando quanto de herança de design dois ativos quaisquer compartilham.

O pedigree é construído para que a estrutura seja interpretável: dois dos quatro modelos da frota atual descendem de um par de designs predecessores **irmãos** e, portanto, carregam herança concentrada, enquanto os outros se baseiam em linhagens disjuntas. O procedimento recupera exatamente isso. Os dois modelos derivados de irmãos (`G3_T1`, `G3_T3`) carregam cada um um coeficiente de endogamia de **F = 0,25**; os outros dois (`G3_T2`, `G3_T4`) têm **F = 0**. O coeficiente de endogamia médio da frota é **0,0417**. Lendo a matriz de parentesco aditivo, o par menos aparentado da frota atual é **`G3_T1` e `G3_T3` (parentesco = 0)** — eles não compartilham ancestralidade e formam o pareamento redundante (N-1) mais forte, o que importa porque `G3_T1` é, ele mesmo, um dos designs mais endogâmicos.

## Fontes de Dados

| Conjunto de Dados | Linhas | Variáveis-Chave | Descrição |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | Um pequeno pedigree de engenharia determinístico de designs de transformadores de subestação ao longo de três gerações de design não sobrepostas (4 plataformas fundadoras, 4 derivados de segunda geração, 4 modelos da frota atual). Cada design não fundador registra os dois designs predecessores distintos (`Parent1`, `Parent2`) dos quais foi derivado. `Sex` marca o papel de liderança do design (M = linhagem mecânica/do núcleo, F = linhagem elétrica/do enrolamento). O pedigree é especificado manualmente — não aleatório — de modo que a estrutura de parentesco é conhecida de antemão e pode ser verificada em relação à saída do procedimento. |

# Quantificando a Herança de Design Compartilhada em uma Frota de Transformadores de Potência

## Por que uma concessionária se importa com "endogamia"

Uma operadora de transmissão e distribuição opera centenas de transformadores de potência de subestação. Para controlar o custo de engenharia e o esforço de qualificação, os fabricantes raramente projetam cada transformador do zero — cada novo modelo **herda** a geometria do núcleo, a topologia dos enrolamentos, os sistemas de isolamento e os designs das buchas de um ou dois modelos predecessores. Ao longo de vários ciclos de aquisição, isso produz uma verdadeira *genealogia de engenharia*: um pedigree de designs.

Essa herança compartilhada é um risco oculto de confiabilidade. Se dois transformadores que protegem a mesma carga (um par redundante **N-1**) descendem do mesmo design ancestral, é provável que um defeito de design latente — um modo de ressonância, um mecanismo de envelhecimento do isolamento, um caminho de descarga disruptiva da bucha — esteja presente em **ambas** as unidades. Uma única causa-raiz pode então eliminar o par redundante simultaneamente: uma *falha de modo comum*.

O **PROC INBREED** foi construído para quantificar exatamente esse tipo de ancestralidade compartilhada. Embora suas origens estejam no melhoramento animal e vegetal, sua matemática — a recursão de parentesco aditivo de Wright — é agnóstica em relação ao domínio: ela mede a fração de herança de design que dois indivíduos compartilham por meio de ancestrais comuns. Mapeamos o vocabulário genético para a engenharia de ativos:

| Conceito do INBREED | Interpretação na concessionária |
|---|---|
| Indivíduo | Um design / modelo de transformador |
| Pai (macho/fêmea) | Um design predecessor do qual foi derivado |
| Geração (CLASS) | Um ciclo de aquisição / design |
| Coeficiente de endogamia *F* | Grau de herança autossimilar dentro de um design |
| Coeficiente de covariância / parentesco | Herança de design compartilhada entre dois designs |
| Par menos aparentado | Melhor pareamento redundante para resiliência N-1 |

## Etapa 1 — Especificar o pedigree de design

Inserimos `DesignLineage` diretamente para que a estrutura de parentesco seja inequívoca. Há três **gerações de design não sobrepostas**:

- **Geração 1** — quatro designs de plataforma fundadores (`G1_P1`-`G1_P4`) sem predecessores (pais em branco).
- **Geração 2** — quatro designs derivados (`G2_D1`-`G2_D4`), cada um projetado a partir de duas plataformas **distintas** da Geração 1. `G2_D1` e `G2_D2` são *irmãos completos* (ambos de `G1_P1` x `G1_P2`); `G2_D3` e `G2_D4` são irmãos completos (ambos de `G1_P3` x `G1_P4`).
- **Geração 3** — quatro modelos da frota atual (`G3_T1`-`G3_T4`). `G3_T1` é construído a partir do par de irmãos `G2_D1` x `G2_D2`, e `G3_T3` a partir do par de irmãos `G2_D3` x `G2_D4`; esses dois designs, portanto, concentram herança. `G3_T2` e `G3_T4` cruzam cada um as duas linhagens disjuntas.

As colunas `ID`, `Parent1` e `Parent2` carregam o pedigree; `Sex` registra qual linhagem de engenharia liderou o design. Uma pequena etapa DATA de acompanhamento apaga o `.` de espaço reservado para que as plataformas fundadoras tenham pais vazios, como o PROC INBREED espera.

In [1]:
DADOS DesignLineage;
   COMPRIMENTO ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   ENTRADA Generation ID $ Parent1 $ Parent2 $ Sex $;
   RÓTULO Generation="Geração" ID="Identificador do Design"
         Parent1="Predecessor 1" Parent2="Predecessor 2"
         Sex="Linhagem Líder";
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
EXECUTAR;

/* Plataformas fundadoras não têm predecessores: apagar os pontos de espaço reservado */
DADOS DesignLineage;
   DEFINIR DesignLineage;
   SE Parent1 = '.' ENTÃO Parent1 = ' ';
   SE Parent2 = '.' ENTÃO Parent2 = ' ';
EXECUTAR;

TÍTULO 'Pedigree de Design dos Transformadores';
PROCEDIMENTO IMPRIMIR DADOS=DesignLineage noobs;
   VARIÁVEL Generation ID Parent1 Parent2 Sex;
EXECUTAR;

                                         Pedigree de Design dos Transformadores                                         


Generation     ID  Parent1  Parent2  Sex
----------  -----  -------  -------  ---
         1  G1_P1                    M
         1  G1_P2                    M
         1  G1_P3                    M
         1  G1_P4                    F
         2  G2_D1  G1_P1    G1_P2    M
         2  G2_D2  G1_P1    G1_P2    F
         2  G2_D3  G1_P3    G1_P4    F
         2  G2_D4  G1_P3    G1_P4    M
         3  G3_T1  G2_D1    G2_D2    M
         3  G3_T2  G2_D1    G2_D3    F
         3  G3_T3  G2_D3    G2_D4    F
         3  G3_T4  G2_D2    G2_D4    M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Pedigree de Design dos Transformadores.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Etapa 2 — Coeficientes de endogamia por geração de design

Executamos o PROC INBREED em **modo multi-geração** nomeando `Generation` na instrução `CLASS`, o que imprime o resumo do pedigree por geração. A instrução `VAR` fornece as três colunas do pedigree na ordem exigida: indivíduo, primeiro predecessor, segundo predecessor.

- A opção **IND** imprime o coeficiente de endogamia *F* de cada design — uma medida de quão concentrada é sua própria herança. Um design construído a partir de dois predecessores intimamente aparentados carrega um *F* alto.
- A opção **MATRIX** imprime a matriz completa de parentesco aditivo para que possamos ler a herança par a par diretamente.
- A opção **AVERAGE** acrescenta o coeficiente de endogamia médio de toda a frota — um único resumo de diversidade de design.

Valores próximos de 0 significam linhagens independentes; *F* aumenta à medida que os dois predecessores de um design se tornam mais intimamente aparentados.

In [2]:
TÍTULO 'Coeficientes de Endogamia por Geração de Design';
PROCEDIMENTO inbreed DADOS=DesignLineage ind average MATRIX;
   VARIÁVEL ID Parent1 Parent2;
   CLASSE Generation;
EXECUTAR;

                                    Coeficientes de Endogamia por Geração de Design                                     


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Geração/Class
--------------------------------------
Geração 1            Members = 4
Geração 2            Members = 4
Geração 3            Members = 4

Inbreeding Coefficients (Geração 1)
--------------------------------------
Identificador do Design  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Geração 2)
--------------------------------------
Identificador do Design  F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficient


NOTE: Option TITLE changed to Coeficientes de Endogamia por Geração de Design.
NOTE: PROC INBREED data=DesignLineage



## Etapa 3 — Coeficientes de covariância salvos para pontuação de risco posterior

Substituindo a visão de endogamia pela opção **COVAR**, obtemos os mesmos relacionamentos relatados como **coeficientes de covariância (parentesco aditivo)**, a forma que uma equipe de gestão de ativos alimentaria em uma pontuação de risco de redundância. A opção **OUTCOV=** captura a matriz completa como um conjunto de dados (`DesignCovar`), em que as colunas `Col1`-`Col12` armazenam o parentesco de cada design com todos os demais (na ordem do pedigree) — pronto para ser unido de volta às atribuições de subestação.

Imprimimos o conjunto de dados de saída para que a matriz salva fique visível junto com a listagem.

In [3]:
TÍTULO 'Coeficientes de Covariância (Parentesco)';
PROCEDIMENTO inbreed DADOS=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   VARIÁVEL ID Parent1 Parent2;
   CLASSE Generation;
EXECUTAR;

TÍTULO 'Matriz de Parentesco (OUTCOV=) Salva para Pontuação de Risco';
PROCEDIMENTO IMPRIMIR DADOS=DesignCovar noobs;
EXECUTAR;

                                        Coeficientes de Covariância (Parentesco)                                        


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Geração/Class
--------------------------------------
Geração 1            Members = 4
Geração 2            Members = 4
Geração 3            Members = 4

Inbreeding Coefficients (Geração 1)
--------------------------------------
Identificador do Design  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Geração 1)
--------------------------------------------------
Identificador do Design     G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.0000
G1_P4


NOTE: Option TITLE changed to Coeficientes de Covariância (Parentesco).
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to Matriz de Parentesco (OUTCOV=) Salva para Pontuação de Risco.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Etapa 4 — Pareamentos menos aparentados para instalações redundantes (N-1)

A matriz `DesignCovar` salva é exatamente o que um estudo de redundância precisa. O parentesco do design *i* com o design *j* está na linha *i*, coluna `Col`*j* (as colunas seguem a ordem do pedigree, de modo que `Col9`-`Col12` correspondem aos quatro modelos da frota atual `G3_T1`-`G3_T4`).

Extraímos as quatro linhas da frota atual do `DesignCovar`, formamos cada par não ordenado da frota atual, anexamos seu coeficiente de parentesco e ordenamos do menos aparentado para o mais aparentado. O objetivo é escolher pares redundantes cuja herança compartilhada seja a **mais baixa** — aqueles que minimizam a chance de que um defeito de design herdado desabilite as duas metades de uma posição protegida N-1.

In [4]:
/* Extrai as quatro linhas da frota atual; Col9..Col12 são os parentescos
   com G3_T1..G3_T4 (ordem das colunas do pedigree). Constrói cada par não ordenado. */
DADOS Pairs;
   DEFINIR DesignCovar;
   ONDE ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   COMPRIMENTO DesignA $12 DesignB $12;
   VETOR g3{4} Col9 Col10 Col11 Col12;
   VETOR nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   RÓTULO DesignA="Design A" DesignB="Design B" Relationship="Parentesco";
   DesignA = ID;
   FAZER j = 1 ATÉ 4;
      DesignB = nm{j};
      SE DesignA < DesignB ENTÃO FAZER;
         Relationship = g3{j};
         SAÍDA;
      FIM;
   FIM;
   MANTER DesignA DesignB Relationship;
EXECUTAR;

PROCEDIMENTO ORDENAR DADOS=Pairs; POR Relationship; EXECUTAR;

TÍTULO 'Pareamentos Redundantes (N-1) Candidatos, do Menos ao Mais Aparentado';
PROCEDIMENTO IMPRIMIR DADOS=Pairs noobs;
   VARIÁVEL DesignA DesignB Relationship;
EXECUTAR;
TÍTULO;

                         Pareamentos Redundantes (N-1) Candidatos, do Menos ao Mais Aparentado                          


DesignA  DesignB  Relationship
-------  -------  ------------
G3_T1    G3_T3               0
G3_T2    G3_T4            0.25
G3_T1    G3_T2           0.375
G3_T1    G3_T4           0.375
G3_T2    G3_T3           0.375
G3_T3    G3_T4           0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Pareamentos Redundantes (N-1) Candidatos, do Menos ao Mais Aparentado.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Interpretando os resultados

**Coeficientes de endogamia (Etapa 2).** Todas as plataformas fundadoras da Geração 1 e todos os derivados da Geração 2 mostram *F* = 0 — por construção, nenhum tem dois predecessores aparentados. Dois modelos da Geração 3 surgem com *F* = 0,25: `G3_T1` (construído a partir do par de irmãos `G2_D1` x `G2_D2`) e `G3_T3` (do par de irmãos `G2_D3` x `G2_D4`). Seus predecessores remontam ao *mesmo* par de plataformas fundadoras, então sua herança é concentrada. Do ponto de vista da confiabilidade, esses são os designs mais expostos a um único defeito herdado, e merecem testes de qualificação independentes adicionais. `G3_T2` e `G3_T4`, que cruzam as duas linhagens disjuntas, têm *F* = 0.

**Matriz de parentesco (Etapa 3).** As entradas fora da diagonal quantificam a herança compartilhada par a par. Os dois pares de irmãos da Geração 2 mostram cada um um parentesco de 0,50 entre si (`G2_D1`-`G2_D2` e `G2_D3`-`G2_D4`), enquanto os derivados de linhagens opostas ficam em 0,00. Os designs endogâmicos da frota atual carregam autorrelacionamentos de 1,25 (= 1 + *F*), visíveis na diagonal. O conjunto de dados `DesignCovar` disponibiliza a matriz completa para ser unida às atribuições de subestação.

**Pareamentos menos aparentados (Etapa 4).** Classificando cada par da frota atual por parentesco, coloca **`G3_T1` e `G3_T3` em primeiro lugar, com parentesco 0,00** — eles descendem de plataformas ancestrais totalmente disjuntas e não compartilham herança de design. Este é o pareamento redundante mais forte, e é especialmente valioso porque `G3_T1` é, ele mesmo, um dos dois designs mais endogâmicos: pareá-lo com um parceiro completamente não aparentado protege contra sua herança concentrada. O próximo melhor par é `G3_T2` e `G3_T4`, em 0,25; os pares restantes ficam em 0,375. O coeficiente de endogamia médio da frota de **0,0417** (impresso pela opção AVERAGE na Etapa 2) resume a diversidade geral de design. As aquisições devem preferir os pares de menor parentesco para posições N-1 e, ao longo do tempo, ampliar a base ancestral — o equivalente, na engenharia de ativos, a evitar um gargalo genético.

**Ressalva.** Estes são dados sintéticos ilustrativos; um estudo de produção construiria o pedigree a partir dos registros reais de revisão de design do fabricante e validaria os escores de parentesco em relação a eventos históricos de falha de modo comum antes de orientar decisões de aquisição.